# Tarea 3 - Punto 2: Verdadera Democracia
## Reparto de Poder Político mediante Algoritmos Genéticos (AGs)
**Asignatura:** Inteligencia Artificial y Mini-robots  

### Enunciado del Problema:
> *"Suponga que usted es el jefe de gobierno y está interesado en que pasen los proyectos de su programa político. Sin embargo, en el congreso conformado por 5 partidos, no es fácil su tránsito, por lo que debe repartir el poder, conformado por ministerios y otras agencias del gobierno, con base en la representación de cada partido. Cada entidad estatal tiene un peso de poder, que es el que se debe distribuir. Suponga que hay 50 curules, distribuya aleatoriamente, con una distribución no uniforme entre los 5 partidos esas curules. Defina una lista de 50 entidades y asígnales aleatoriamente un peso político de 1 a 100 puntos. Cree una matriz de poder para repartir ese poder, usando AGs."*

--- 
### 1. Importación de Librerías y Configuración Inicial

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

# Fijar semillas para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Entorno inicializado correctamente.")

--- 
### 2. Definición del Escenario: 5 Partidos, 50 Curules y 50 Entidades Estatales

In [ ]:
PARTIDOS = [
    "Partido Renovación Democrática (PRD)",
    "Alianza Social Progresista (ASP)",
    "Fuerza Conservadora Unida (FCU)",
    "Movimiento Centro Plural (MCP)",
    "Coalición Verde & Regional (CVR)"
]

NUM_PARTIDOS = len(PARTIDOS)
TOTAL_CURULES = 50
TOTAL_ENTIDADES = 50

NOMBRES_ENTIDADES = [
    "Ministerio de Hacienda y Crédito Público", "Ministerio del Interior",
    "Ministerio de Defensa Nacional", "Ministerio de Justicia y del Derecho",
    "Ministerio de Salud y Protección Social", "Ministerio de Educación Nacional",
    "Ministerio de Minas y Energía", "Ministerio de Transporte",
    "Ministerio de TIC", "Ministerio del Trabajo",
    "Ministerio de Agricultura", "Ministerio de Ambiente",
    "Ministerio de Comercio e Industria", "Ministerio de Vivienda",
    "Ministerio de Cultura", "Ministerio del Deporte",
    "Ministerio de Ciencia y Tecnología", "Ministerio de Igualdad y Equidad",
    "DNP - Planeación Nacional", "DANE - Estadísticas",
    "DAFP - Función Pública", "DAPRE - Presidencia",
    "DPS - Prosperidad Social", "DIAN - Aduanas e Impuestos",
    "Superintendencia Financiera", "Superintendencia Nacional de Salud",
    "Superintendencia de Sociedades", "Superintendencia de Industria y Comercio",
    "Superintendencia de Transporte", "Superintendencia de Servicios Públicos",
    "Superintendencia de Notariado", "Agencia Nacional de Infraestructura (ANI)",
    "Agencia Nacional de Minería (ANM)", "Agencia Nacional de Hidrocarburos (ANH)",
    "Agencia Nacional de Tierras (ANT)", "Agencia de Desarrollo Rural (ADR)",
    "Agencia Nacional de Seguridad Vial", "Agencia Nacional de Licencias (ANLA)",
    "Agencia Presidencial de Cooperación", "Instituto Nacional de Vías (INVIAS)",
    "Bienestar Familiar (ICBF)", "SENA",
    "Instituto Geográfico Agustín Codazzi (IGAC)", "IDEAM - Meteorología",
    "INVIMA - Vigilancia Medicamentos", "ICA - Instituto Agropecuario",
    "Fondo de Adaptación", "UNGRD - Gestión del Riesgo",
    "UPRA - Planificación Rural", "Unidad de Víctimas"
]

def generar_curules_no_uniforme(total_curules=50, num_partidos=5):
    pesos = np.random.exponential(scale=2.0, size=num_partidos) + 0.5
    proporciones = pesos / np.sum(pesos)
    curules = np.ones(num_partidos, dtype=int)
    restantes = total_curules - num_partidos
    dist_adicional = np.random.multinomial(restantes, proporciones)
    curules += dist_adicional
    return curules

def generar_pesos_entidades(num_entidades=50):
    return np.random.randint(1, 101, size=num_entidades)

curules = generar_curules_no_uniforme(TOTAL_CURULES, NUM_PARTIDOS)
pesos_entidades = generar_pesos_entidades(TOTAL_ENTIDADES)
poder_total = np.sum(pesos_entidades)

print(f"Distribución de Curules: {curules} (Total: {np.sum(curules)})")
print(f"Poder Total Estatal: {poder_total} puntos")

--- 
### 3. Implementación del Algoritmo Genético (AG)

In [ ]:
class AlgoritmoGeneticoPoder:
    def __init__(
        self,
        curules,
        pesos_entidades,
        poblacion_tam=150,
        generaciones=350,
        prob_cruce=0.85,
        prob_mutacion=0.035,
        k_torneo=3,
        elitismo_tam=4
    ):
        self.curules = np.array(curules)
        self.num_partidos = len(curules)
        self.pesos_entidades = np.array(pesos_entidades)
        self.num_entidades = len(pesos_entidades)
        self.poblacion_tam = poblacion_tam
        self.generaciones = generaciones
        self.prob_cruce = prob_cruce
        self.prob_mutacion = prob_mutacion
        self.k_torneo = k_torneo
        self.elitismo_tam = elitismo_tam
        
        self.poder_total = np.sum(self.pesos_entidades)
        self.proporcion_curules = self.curules / np.sum(self.curules)
        self.poder_objetivo = self.proporcion_curules * self.poder_total
        
        self.historial_mejor_fitness = []
        self.historial_mejor_error = []
        self.historial_promedio_fitness = []
        self.mejor_individuo = None
        self.mejor_fitness = -1.0
        self.mejor_error = float('inf')

    def calcular_poder_asignado(self, cromosoma):
        poder_partidos = np.zeros(self.num_partidos)
        for entidad_idx, partido_idx in enumerate(cromosoma):
            poder_partidos[partido_idx] += self.pesos_entidades[entidad_idx]
        return poder_partidos

    def calcular_fitness(self, cromosoma):
        poder_asignado = self.calcular_poder_asignado(cromosoma)
        error_total = np.sum(np.abs(poder_asignado - self.poder_objetivo))
        fitness = 1000.0 / (1.0 + error_total)
        return fitness, error_total

    def inicializar_poblacion(self):
        return [np.random.randint(0, self.num_partidos, size=self.num_entidades) for _ in range(self.poblacion_tam)]

    def seleccion_torneo(self, poblacion, fitnesses):
        seleccionados_indices = np.random.choice(len(poblacion), size=self.k_torneo, replace=False)
        mejor_idx = seleccionados_indices[0]
        mejor_fit = fitnesses[mejor_idx]
        for idx in seleccionados_indices[1:]:
            if fitnesses[idx] > mejor_fit:
                mejor_fit = fitnesses[idx]
                mejor_idx = idx
        return poblacion[mejor_idx].copy()

    def cruce_uniforme(self, padre1, padre2):
        if random.random() > self.prob_cruce:
            return padre1.copy(), padre2.copy()
        mascara = np.random.rand(self.num_entidades) < 0.5
        hijo1 = np.where(mascara, padre1, padre2)
        hijo2 = np.where(mascara, padre2, padre1)
        return hijo1, hijo2

    def mutar(self, cromosoma):
        for i in range(self.num_entidades):
            if random.random() < self.prob_mutacion:
                cromosoma[i] = random.randint(0, self.num_partidos - 1)
        return cromosoma

    def ejecutar(self):
        poblacion = self.inicializar_poblacion()
        for gen in range(self.generaciones):
            evaluaciones = [self.calcular_fitness(ind) for ind in poblacion]
            fitnesses = [ev[0] for ev in evaluaciones]
            errores = [ev[1] for ev in evaluaciones]
            
            idx_mejor = int(np.argmax(fitnesses))
            if fitnesses[idx_mejor] > self.mejor_fitness:
                self.mejor_fitness = fitnesses[idx_mejor]
                self.mejor_error = errores[idx_mejor]
                self.mejor_individuo = poblacion[idx_mejor].copy()
                
            self.historial_mejor_fitness.append(self.mejor_fitness)
            self.historial_mejor_error.append(self.mejor_error)
            self.historial_promedio_fitness.append(np.mean(fitnesses))
            
            indices_ordenados = np.argsort(fitnesses)[::-1]
            nueva_poblacion = [poblacion[i].copy() for i in indices_ordenados[:self.elitismo_tam]]
            
            while len(nueva_poblacion) < self.poblacion_tam:
                padre1 = self.seleccion_torneo(poblacion, fitnesses)
                padre2 = self.seleccion_torneo(poblacion, fitnesses)
                hijo1, hijo2 = self.cruce_uniforme(padre1, padre2)
                nueva_poblacion.append(self.mutar(hijo1))
                if len(nueva_poblacion) < self.poblacion_tam:
                    nueva_poblacion.append(self.mutar(hijo2))
                    
            poblacion = nueva_poblacion
        return self.mejor_individuo, self.mejor_fitness, self.mejor_error

    def obtener_matriz_poder(self, cromosoma=None):
        if cromosoma is None:
            cromosoma = self.mejor_individuo
        matriz = np.zeros((self.num_partidos, self.num_entidades), dtype=int)
        for entidad_idx, partido_idx in enumerate(cromosoma):
            matriz[partido_idx, entidad_idx] = 1
        return matriz

--- 
### 4. Ejecución y Análisis de Resultados

In [ ]:
ag = AlgoritmoGeneticoPoder(
    curules=curules,
    pesos_entidades=pesos_entidades,
    poblacion_tam=150,
    generaciones=350,
    prob_cruce=0.85,
    prob_mutacion=0.035,
    k_torneo=3,
    elitismo_tam=4
)

mejor_cromosoma, mejor_fitness, mejor_error = ag.ejecutar()
poder_asignado = ag.calcular_poder_asignado(mejor_cromosoma)
matriz_poder = ag.obtener_matriz_poder()
entidades_por_partido = np.sum(matriz_poder, axis=1)

print("=" * 95)
print(f"{'Partido':<38} | {'Curules':<7} | {'% Curules':<9} | {'Poder Obj':<9} | {'Poder Asig':<10} | {'% Poder':<7} | {'Error':<5} | {'# Ent.'}")
print("=" * 95)
for i in range(NUM_PARTIDOS):
    porc_c = (curules[i] / TOTAL_CURULES) * 100
    p_obj = ag.poder_objetivo[i]
    p_asig = poder_asignado[i]
    porc_p = (p_asig / poder_total) * 100
    err = abs(p_asig - p_obj)
    num_ent = entidades_por_partido[i]
    print(f"{PARTIDOS[i]:<38} | {curules[i]:<7} | {porc_c:>7.1f}% | {p_obj:>9.1f} | {p_asig:>10.1f} | {porc_p:>6.1f}% | {err:>5.1f} | {num_ent:>5}")
print("=" * 95)

--- 
### 5. Gráficos de Visualización

In [ ]:
# Gráfica 1: Curva de Convergencia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
ax1.plot(ag.historial_mejor_error, color='#c0392b', lw=2.2, label='Mejor Error Absoluto')
ax1.set_title('Convergencia del Error Residual', fontweight='bold')
ax1.set_xlabel('Generación')
ax1.set_ylabel('Error Absoluto Acumulado')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.plot(ag.historial_mejor_fitness, color='#27ae60', lw=2.2, label='Mejor Fitness')
ax2.plot(ag.historial_promedio_fitness, color='#2980b9', linestyle='--', label='Fitness Promedio')
ax2.set_title('Evolución del Fitness', fontweight='bold')
ax2.set_xlabel('Generación')
ax2.set_ylabel('Fitness (Aptitud)')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 2: Comparativa de Proporcionalidad
porc_curules = ag.proporcion_curules * 100
porc_poder_asignado = (poder_asignado / ag.poder_total) * 100

fig, ax = plt.subplots(figsize=(10, 5))
indices = np.arange(ag.num_partidos)
ancho = 0.35

b1 = ax.bar(indices - ancho/2, porc_curules, ancho, label='Curules (%)', color='#2980b9', alpha=0.85)
b2 = ax.bar(indices + ancho/2, porc_poder_asignado, ancho, label='Poder Asignado (%)', color='#d35400', alpha=0.85)
ax.set_title('Comparativa de Proporcionalidad: Curules vs Poder Asignado', fontweight='bold')
ax.set_xticks(indices)
ax.set_xticklabels([f"P{i+1}\n({c} curules)" for i, c in enumerate(ag.curules)])
ax.set_ylabel('Porcentaje (%)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 3: Matriz de Poder (Heatmap 5x50)
matriz_pesos = matriz_poder * ag.pesos_entidades.reshape(1, -1)
fig, ax = plt.subplots(figsize=(16, 4.5))
cax = ax.imshow(matriz_pesos, cmap='YlGnBu', aspect='auto')
ax.set_title('Matriz de Distribución de Poder Estatal (5 Partidos x 50 Entidades)', fontweight='bold')
ax.set_xlabel('Entidad Estatal (0 a 49)', fontweight='bold')
ax.set_ylabel('Partidos Políticos', fontweight='bold')
ax.set_yticks(np.arange(ag.num_partidos))
ax.set_yticklabels([f"P{i+1}: {PARTIDOS[i][:15]}..." for i in range(ag.num_partidos)])
cbar = fig.colorbar(cax, orientation='horizontal', pad=0.28, shrink=0.7)
cbar.set_label('Peso Político Asignado (0 = No asignado, >0 = Puntos de poder)')
plt.tight_layout()
plt.show()